In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("Upi_trans_silver").getOrCreate()

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_date", "")

In [0]:
schema = StructType([
    StructField("txn_id", StringType(), True),
    StructField("rrn", StringType(), True),
    StructField("payer_vpa", StringType(), True),
    StructField("payee_vpa", StringType(), True),
    StructField("payer_bank", StringType(), True),
    StructField("payee_bank", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("txn_type", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("status", StringType(), True),
    StructField("npci_response_code", StringType(), True),
    StructField("initiated_timestamp", StringType(), True)
    ])

In [0]:
## INflating the json, casting and aliasing

silver_inflated =(spark.readStream.table("workspace.upi_schema.upi_transactions_bronze")
    .select(
    col("key"),from_json(col("value"),schema).alias("fields"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
    )
    .select(
    col("key").cast("string").alias("key"),
    col("fields.txn_id").cast("string").alias("txn_id"),
    col("fields.rrn").cast("string").alias("rrn"),
    col("fields.payer_vpa").cast("string").alias("payer_vpa"),
    col("fields.payee_vpa").cast("string").alias("payee_vpa"),
    col("fields.payer_bank").cast("string").alias("payer_bank"),
    col("fields.payee_bank").cast("string").alias("payee_bank"),
    col("fields.amount").cast("decimal(10,2)").alias("amount"),
    col("fields.txn_type").cast("string").alias("txn_type"),
    col("fields.channel").cast("string").alias("channel"),
    col("fields.status").cast("string").alias("status"),
    col("fields.npci_response_code").cast("string").alias("npci_response_code"),
    col("fields.initiated_timestamp").cast("timestamp").alias("initiated_timestamp"),
    col("topic").cast("string").alias("topic"),
    col("partition").cast("int").alias("partition"),
    col("offset").cast("long").alias("offset"),
    col("timestamp").cast("timestamp").alias("timestamp"),
    col("timestampType").cast("int").alias("timestampType")
    )
)

bronze_query = (
                silver_inflated.writeStream
                .format("delta")
                .outputMode("append")
                .option("checkpointLocation", "/Volumes/workspace/upi_schema/checkpoints/SILVER LAYER/Bronze_Inflated/")
                .trigger(availableNow = True)
                .queryName("bronze_query_inflated")
                .toTable("workspace.upi_schema.upi_transactions_bronze_inflated")
                
                )



In [0]:

silver_inflated = (silver_inflated.withColumn("late_flag",when(col("initiated_timestamp") < current_timestamp() - expr("interval 2 minute"),lit(1)).otherwise(lit(0)))
        .withColumn("txn_date",date_format(col("initiated_timestamp"),"yyyy-MM-dd"))
       .withColumn("txn_hour",date_format(col("initiated_timestamp"),"HH")))

column_validations = (
                        col("txn_id").isNotNull() 
                        & col("rrn").isNotNull() 
                        & col("amount").isNotNull() 
                        & col("initiated_timestamp").isNotNull()
                        & ((col("status") == "PENDING")
                            | col("npci_response_code").isNotNull()
                          )
                        & ((col("amount") > 0) & (col("amount")<100000))
                        & upper(col("status")).isin(["SUCCESS", "FAILED", "PENDING"])
                        & col("channel").isin(["APP", "QR", "COLLECT"])
                        & col("txn_type").isin(["P2P", "P2M"])
                    )

run_id = dbutils.widgets.get("run_id")

quarantine_df = silver_inflated.filter(~column_validations).withColumn("run_id",lit(run_id))

quarantine_query = (quarantine_df.writeStream.format("delta")
                        .outputMode("append")
                        .option("checkpointLocation","/Volumes/workspace/upi_schema/checkpoints/SILVER LAYER/Quarantine/")
                        .trigger(availableNow = True)
                        .queryName("quarantine_query")
                        .toTable("workspace.upi_schema.upi_transactions_quarantine")
)




In [0]:

## Validating wheteher important columns are there or not
column_validated_df = silver_inflated.filter(column_validations)


column_standardizations_df = (column_validated_df
                              .withColumn("status",upper(col("status")))
                              .withColumn("txn_type",upper(col("txn_type")))
                              .withColumn("channel",upper(col("channel")))
                              .withColumn("payer_vpa",lower(col("payer_vpa")))
                              .withColumn("payee_vpa",lower(col("payee_vpa")))
                              )

query_silver = (column_standardizations_df.writeStream
               .format("delta")
               .option("checkpointLocation", "/Volumes/workspace/upi_schema/checkpoints/SILVER LAYER/Silver_Standardized/")
               .outputMode("append")
               .trigger(availableNow = True)
               .queryName("inflated_query")
               .toTable("workspace.upi_schema.upi_transactions_standardized")
               )

In [0]:
silver_dedup = (spark.readStream.table("workspace.upi_schema.upi_transactions_standardized")
                          .withWatermark("initiated_timestamp", "20 minutes")
                          .dropDuplicatesWithinWatermark(["txn_id"])
                          .withColumn("silver_processed_timestamp",current_timestamp())
)
query_dedup = (silver_dedup.writeStream
               .format("delta")
               .option("checkpointLocation","/Volumes/workspace/upi_schema/checkpoints/SILVER LAYER/Silver_Dedup/")
               .outputMode("append")
               .trigger(availableNow = True)
               .queryName("dedup_query")
               .toTable("workspace.upi_schema.upi_transactions_silver")
)

In [0]:
%sql
create or replace view workspace.upi_schema.upi_stream_silver_view as
select * from workspace.upi_schema.upi_transactions_silver where status ='PENDING' or status ='SUCCESS'

> # **Settlement_data enrichments**

In [0]:
settlement_bronze_df = spark.readStream.table("workspace.upi_schema.bronze_settlement_data").drop("_rescued_data")

In [0]:
column_validations = (
                        col("txn_id").isNotNull() 
                        & col("rrn").isNotNull() 
                        & col("settlement_amount").isNotNull() 
                        & col("settled_timestamp").isNotNull()
                        & ((col("settlement_amount") > 0) & (col("settlement_amount")<100000))
                        & upper(col("settlement_status")).isin(["SETTLED", "PENDING"])
                    )

In [0]:
settlement_bronze_df_dedup = (settlement_bronze_df.filter(column_validations)
                        .withColumn("settlement_status",upper(col("settlement_status")))
                        .withWatermark("settled_timestamp", "1 day")
                        .dropDuplicatesWithinWatermark(["txn_id"])
                        )
        
settlement_silver_query_dedup = (settlement_bronze_df_dedup.writeStream
                           .format("delta")
                           .outputMode("append")
                           .option("checkpointLocation","/Volumes/workspace/upi_schema/checkpoints/SILVER LAYER/Settlement_Silver/")
                           .trigger(availableNow = True)
                           .queryName("settlement_silver_query_dedup")
                           .toTable("workspace.upi_schema.settlement_silver")
                            )


In [0]:
      
settlement_silver_query = (settlement_bronze_df.writeStream
                           .format("delta")
                           .outputMode("append")
                           .option("checkpointLocation","/Volumes/workspace/upi_schema/checkpoints/SILVER LAYER/Settlement_Silver_for_count/")
                           .trigger(availableNow = True)
                           .queryName("settlement_silver_query")
                           .toTable("workspace.upi_schema.settlement_silver_for_count")
                            )


In [0]:
settlement_quarantine = settlement_bronze_df.filter(~column_validations)

query_quarantine = (settlement_quarantine.writeStream
                     .format("delta")
                     .outputMode("append")
                     .option("checkpointLocation","/Volumes/workspace/upi_schema/checkpoints/SILVER LAYER/Settlement_Quarantine/")
                     .trigger(availableNow = True)
                     .queryName("settlement_quarantine_query")
                     .toTable("workspace.upi_schema.settlement_quarantine")
)

> # Metrics

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
txn_silver_count=0
query_dedup.awaitTermination()
for batch in query_dedup.recentProgress:
    txn_silver_count += batch["sources"][0]["numInputRows"]
data =[run_id,run_date,txn_silver_count]
cols=["run_id","run_date","txn_silver_count"]

txn_silver_count_df = spark.createDataFrame([data],cols)

txn_silver_count_df.createOrReplaceTempView("txn_silver_temp")

spark.sql("""
    MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
    USING txn_silver_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.txn_silver_count = s.txn_silver_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, txn_silver_count) VALUES (s.run_id,s.run_date, s.txn_silver_count)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id=dbutils.widgets.get("run_id")
txn_silver_reject_count=0
txn_silver_reject_count=spark.table("workspace.upi_schema.upi_transactions_quarantine").filter(col("run_id")==lit(run_id)).count()

print(txn_silver_reject_count)

0


In [0]:


run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
txn_silver_reject_count=0
quarantine_query.awaitTermination()
txn_silver_reject_count=spark.table("workspace.upi_schema.upi_transactions_quarantine").filter(col("run_id")==lit(run_id)).count()
data =[run_id,run_date,txn_silver_reject_count]
cols=["run_id","run_date","txn_silver_reject_count"]

txn_silver_reject_count_df = spark.createDataFrame([data],cols)

txn_silver_reject_count_df.createOrReplaceTempView("txn_silver_reject_temp")

spark.sql("""
    MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
    USING txn_silver_reject_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.txn_silver_reject_count = s.txn_silver_reject_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, txn_silver_reject_count) VALUES (s.run_id, s.run_date,s.txn_silver_reject_count)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
settlement_silver_count=0
settlement_silver_query_dedup.awaitTermination()
for batch in settlement_silver_query_dedup.recentProgress:
    settlement_silver_count += batch["sources"][0]["numInputRows"]
data =[run_id,run_date,settlement_silver_count]
cols=["run_id","run_date","settlement_silver_count"]

settlement_silver_count_df = spark.createDataFrame([data],cols)

settlement_silver_count_df.createOrReplaceTempView("settlement_silver_count_temp")

spark.sql("""
    MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
    USING settlement_silver_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.settlement_silver_count = s.settlement_silver_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, settlement_silver_count) VALUES (s.run_id,s.run_date, s.settlement_silver_count)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
settlement_silver_rejected_count=0
query_quarantine.awaitTermination()
for batch in query_quarantine.recentProgress:
    settlement_silver_rejected_count += batch["sources"][0]["numInputRows"]
data =[run_id,run_date,settlement_silver_rejected_count]
cols=["run_id","run_date","settlement_silver_rejected_count"]

settlement_silver_rejected_count_df = spark.createDataFrame([data],cols)

settlement_silver_rejected_count_df.createOrReplaceTempView("settlement_silver_rejected_count_temp")

spark.sql("""
    MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
    USING settlement_silver_rejected_count_temp s
    ON t.run_id = s.run_id
    WHEN MATCHED THEN UPDATE SET t.settlement_silver_rejected_count = s.settlement_silver_rejected_count
    WHEN NOT MATCHED THEN INSERT (run_id,run_date, settlement_silver_rejected_count) VALUES (s.run_id,s.run_date, s.settlement_silver_rejected_count)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")

txn_bronze_count=0
for batch in bronze_query.recentProgress:
    txn_bronze_count += batch["sources"][0]["numInputRows"]
    
if txn_bronze_count>0:
    txn_rejection_rate=txn_silver_reject_count/txn_bronze_count
else:
    txn_rejection_rate=0
data =[run_id,run_date,txn_rejection_rate]
cols=["run_id","run_date","txn_rejection_rate"]

txn_rejection_rate_df = spark.createDataFrame([data],cols)

txn_rejection_rate_df.createOrReplaceTempView("txn_rejection_rate_temp")

spark.sql("""
          MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
          USING txn_rejection_rate_temp s
          on t.run_id=s.run_id
          WHEN MATCHED THEN UPDATE SET t.txn_rejection_rate=s.txn_rejection_rate
          WHEN NOT MATCHED then INSERT (run_id,run_date,txn_rejection_rate) VALUES(s.run_id,s.run_date,s.txn_rejection_rate)
          """)



DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")

settlement_bronze_count=0
for batch in settlement_silver_query.recentProgress:
    settlement_bronze_count += batch["sources"][0]["numInputRows"]

if settlement_bronze_count>0:
    settlement_rejection_rate = settlement_silver_rejected_count/settlement_bronze_count
else:
    settlement_rejection_rate = 0
    
data =[run_id,run_date,settlement_rejection_rate]
cols=["run_id","run_date","settlement_rejection_rate"]

settlement_rejection_rate_df = spark.createDataFrame([data],cols)
settlement_rejection_rate_df.createOrReplaceTempView("settlement_rejection_rate_temp")

spark.sql("""
          MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
          USING settlement_rejection_rate_temp s
          on t.run_id=s.run_id
          WHEN MATCHED THEN UPDATE SET t.settlement_rejection_rate=s.settlement_rejection_rate
          WHEN NOT MATCHED then INSERT (run_id,run_date,settlement_rejection_rate) VALUES(s.run_id,s.run_date,s.settlement_rejection_rate)
          """)



DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]